# Module 2.1: Operating System Evaluation, Ubuntu Server 24.04 LTS Install & Internal NVMe Data Mount

## Homelab Expansion (Chapter 2) - Step 2: Operating System & Headless Laptop Tuning

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why wiping the laptop's existing **unlicensed Windows 11** installation in favor of a headless Linux server OS is mandatory for 24/7 GPU homelab stability.
2. Evaluate Linux server operating systems (`Ubuntu Server 24.04 LTS`, `Debian 12 Bookworm`, and `Proxmox VE 8`) for an ASUS ROG laptop with hybrid AMD/Intel + NVIDIA graphics.
3. Flash a bootable `Ubuntu Server 24.04 LTS (64-bit x86_64)` USB installer and safely wipe **only the stock Windows OS NVMe drive (Slot 1)** while keeping the **WD Blue SN5000 (Slot 2)** completely untouched.
4. Persistently mount the internal **WD Blue SN5000** at `/mnt/homelab_storage` via `UUID` in `/etc/fstab` on `homelab-gpu` and verify your Chapter 1 Immich photo library and `immich_migration_dump.sql.gz` arrived 100% intact.
5. Record your permanent Node 2 credentials for use across the rest of the homelab curriculum.

## 2.1.1 Conceptual Background: Why Wipe Unlicensed Windows 11 & Dual-NVMe Separation

Right now, your 2019 ROG Zephyrus's stock Slot 1 drive is running an **unlicensed copy of Windows 11**, while Slot 2 holds your **WD Blue SN5000** (`ext4`) migrated from Chapter 1. Separating the **OS Drive (Slot 1)** from your **Dedicated Data/Database Drive (Slot 2 at `/mnt/homelab_storage`)** is standard enterprise server practice: you can upgrade or reinstall Ubuntu on Slot 1 at any time without ever risking your Immich photo library or persistent data!

### 1. Why Unlicensed Windows 11 Fails as a 24/7 Homelab Server
- **3.5 GB+ Idle RAM & VRAM Overhead:** Windows 11 Desktop (`dwm.exe`, Windows Defender, telemetry, and WSL2's `vmmem` hypervisor VM) consumes **3.5 GB to 5.0 GB of RAM** and **400 MB+ of GPU VRAM** before a single container is even launched.
- **Forced Unattended Reboots & Activation Nags:** Windows 11 automatically restarts outside "Active Hours" for cumulative updates and driver pushes, killing your Docker stacks or getting stuck at a login screen waiting for user interaction. Unlicensed installs also lock out personalization/group policy controls and display persistent activation watermarks.
- **WSL2 Network & GPU Paravirtualization Quirks:** Docker Desktop on Windows runs inside a lightweight Hyper-V utility VM. Exposing container ports to your LAN or Tailscale mesh requires brittle `netsh interface portproxy` rules, and GPU sleep states are poorly managed.

### 2. Operating System Candidate Comparison for a 2019 ROG Zephyrus (`x86_64` + NVIDIA GPU)

| Operating System | Recommended Level | Idle RAM | NVIDIA CUDA / Docker Support | ASUS ROG Laptop Compatibility (`asus-nb-wmi`) | Verdict |
| :--- | :---: | :---: | :--- | :--- | :--- |
| **Ubuntu Server 24.04 LTS (Noble Numbat)** | 🥇 **Primary Choice** | **~320 MB** | **Best-in-Class:** Built-in `ubuntu-drivers` auto-detects NVIDIA GTX 1660 Ti / RTX 20-series GPUs and installs tested headless drivers (`nvidia-driver-550-server`) in one command. Official NVIDIA Container Toolkit apt repo support. | Modern Linux `6.8+` HWE kernel includes native `asus-nb-wmi` thermal profiles, fan curve telemetry, and Realtek 2.5GbE USB/PCIe drivers out of the box. | **Ideal choice:** Zero desktop bloat, rock-solid 5-year LTS support, and effortless NVIDIA GPU passthrough to Docker Compose. |
| **Debian 12 (Bookworm) Netinst** | 🥈 **Runner-Up** | **~180 MB** | Requires manually enabling `non-free-firmware` and `contrib` repos; older default `6.1` kernel lacks newer ASUS ROG WMI power-profile patches unless backports are configured. | Good, but Wi-Fi / USB Ethernet chipsets often require non-free firmware ISOs during install. | Great if you want strict parity with Raspberry Pi OS, but takes extra manual driver work for NVIDIA + ASUS hardware. |
| **Proxmox VE 8.x (Type-1 Hypervisor)** | 🥉 **Alternative (If You Need VMs)** | **~1.2 GB** | Passing a laptop's **hybrid Optimus GPU** (where the display is wired to AMD/Intel iGPU and the NVIDIA dGPU is a render-only 3D controller) into a VM via PCIe VFIO IOMMU passthrough is notoriously complex and locks the GPU to one VM. | Strips out standard laptop power management utilities unless configured manually on the Debian host shell. | Overkill if your primary goal is running Docker containers (Immich, UniFi, Paperless-ngx, Ollama) with shared GPU access. |

> 💡 **Engineering Conclusion:** By installing **Ubuntu Server 24.04 LTS (Headless, No GUI)** directly on bare metal on Slot 1—and mounting your WD Blue SN5000 in Slot 2 at `/mnt/homelab_storage`—your 16GB/32GB ROG Zephyrus boots in 8 seconds, uses less than **350 MB of RAM** at idle, keeps the NVIDIA GPU in its ultra-low-power **P8 idle state (~3W–5W)** when idle, and shares the GPU seamlessly across Immich CUDA ML, NVENC transcoding, and future AI containers!

## 2.1.2 Step-by-Step Laboratory Guide

### Task 2.1.1: Download Ubuntu Server 24.04 LTS & Flash USB Installer
1. On your primary workstation, download the official **Ubuntu Server 24.04 LTS (64-bit PC `amd64` ISO)** from [ubuntu.com/download/server](https://ubuntu.com/download/server).
2. Download and open **[BalenaEtcher](https://etcher.balena.io/)** (macOS/Windows/Linux) or **[Rufus](https://rufus.ie/)** (Windows).
3. Insert an empty **8GB+ USB 3.0 flash drive** into your workstation.
4. Select the `ubuntu-24.04.x-live-server-amd64.iso` image, select your USB drive, and click **Flash!**.

### Task 2.1.2: Boot the Installer & Wipe *Only* the Stock Windows 11 NVMe Drive
> ⚠️ **DUAL-NVME DATA-SAFETY CHECK:** You now have **two internal NVMe drives** inside the ROG Zephyrus (`nvme0n1` and `nvme1n1`). The Linux kernel enumerates PCIe NVMe namespaces dynamically—never assume `nvme0n1` is the stock drive! **Always verify the drive model string and size** before selecting `Use an entire disk`:
> - **WIPE THIS ONE:** Your factory stock laptop SSD (e.g., Intel 660p / Micron / Samsung OEM, holding the old Windows NTFS partitions).
> - **DO NOT TOUCH THIS ONE:** **`WD Blue SN5000`** (holding your intact Chapter 1 `ext4` Immich photo library and `immich_migration_dump.sql.gz`).

1. Plug your **Gigabit Ethernet cable** into the ROG Zephyrus (either into the native RJ45 port on `GX502` or via your USB 3.0 Ethernet adapter on `GA502`/`GU502`) connected to your home router for initial OS installation.
2. Insert the flashed Ubuntu Server USB drive into a USB port on the ROG Zephyrus.
3. Power on the laptop and rapidly tap **`ESC`** (the ASUS ROG One-Time Boot Menu key) until the **Please select boot device** popup appears.
4. Select **`UEFI: <Your USB Drive Name>, Partition 1`** and press `Enter`, then select **Try or Install Ubuntu Server**.
5. Walk through the text-based `subiquity` installer using your arrow keys, `Space`, and `Enter`:
   - **Language & Keyboard:** `English (US)`.
   - **Installation Type:** Select **`Ubuntu Server`** (standard, not minimized, so diagnostic utilities like `htop`, `curl`, and `tmux` are pre-installed).
   - **Network Connections:** Confirm your wired Ethernet interface (`enp...` or `enx...`) has automatically acquired an IPv4 lease via DHCP.
   - **Guided Storage Configuration (Wiping Windows 11 Only):**
     - Select **`Use an entire disk`** and carefully choose **ONLY the stock Windows NVMe SSD** (double-check it is **NOT** `WD Blue SN5000`!).
     - **Uncheck** `Set up this disk as an LVM group` (creates a clean, single full-disk `ext4` root partition `/` on Slot 1, leaving Slot 2 completely untouched).
     - On the File System Summary screen, double-check that **`WD Blue SN5000`** appears untouched under **Available Devices** with zero destructive actions queued against it, then select `Continue`.
   - **Profile Setup:**
     - **Your name:** `pi` *(or your preferred admin name)*
     - **Your server's name (Hostname):** `homelab-gpu`
     - **Pick a username:** `pi` *(Keeping the admin username identical to `homelab-pi` simplifies SSH config and scripts across your cluster!)*
     - **Password:** Choose a strong administrator password.
   - **Ubuntu Pro Upgrade:** Select `Skip for now`.
   - **SSH Setup:** Press `Space` to check **`[X] Install OpenSSH server`**.
   - **Featured Server Snaps:** **Do NOT check anything here!** *(CRITICAL: Never install `docker` from the Snap list during Ubuntu setup—Snap Docker has strict confinement bugs with NVIDIA GPUs and custom bind mounts. We will install official upstream `docker-ce` via `apt` in Module 4.1).*
6. Wait ~3 minutes for installation to finish, select **`Reboot Now`**, remove the USB flash drive when prompted, and press `Enter`.

---

## 📝 Node 2 Credential Record Sheet

*Record the exact credentials configured for your ROG Zephyrus GPU node below:*

> ### 🔑 Node 2 (`homelab-gpu`) Credentials
> - **Hostname:** `homelab-gpu`
> - **Admin Username:** `pi`
> - **Password:** `[Write your password here]`
> - **Local mDNS Address:** `homelab-gpu.local`
> - **Target Reserved Static IP (Step 3):** `10.0.0.201` *(paired alongside `homelab-pi` at `10.0.0.200`)*

### Task 2.1.3: Verify Clean Bare-Metal Boot & Persistently Mount the Internal WD Blue SN5000
Once the laptop reboots into the Ubuntu Server login prompt (`homelab-gpu login:`), connect via SSH from your workstation (`ssh pi@homelab-gpu.local`) and verify the OS, GPU, and both internal NVMe drives:
```bash
# 1. Verify OS version, kernel, and x86_64 architecture
hostnamectl

# 2. Confirm BOTH internal NVMe drives appear (Slot 1 OS disk + Slot 2 WD Blue SN5000 ext4 partition)
lsblk -o NAME,MODEL,SIZE,FSTYPE,UUID,MOUNTPOINT

# 3. Verify ultra-low headless RAM consumption (~300MB used out of 16GB/32GB!) and NVIDIA PCIe GPU
free -h
lspci | grep -E "VGA|3D"
```

Now mount your **WD Blue SN5000** (`ext4`) persistently at `/mnt/homelab_storage` via its immutable `UUID` (the exact same mount path used in Chapter 1!) and confirm all your migrated Chapter 1 photos and SQL dump are present:
```bash
# 4. Create the standardized mount point
sudo mkdir -p /mnt/homelab_storage

# 5. Copy the UUID of your WD Blue SN5000 ext4 partition from `lsblk` above and paste it below:
SN5000_UUID="PASTE_YOUR_SN5000_PARTITION_UUID_HERE"

sudo cp /etc/fstab /etc/fstab.backup
echo "UUID=${SN5000_UUID}  /mnt/homelab_storage  ext4  defaults,noatime,nofail  0  2" | sudo tee -a /etc/fstab

# 6. Reload systemd mounts, mount the drive, and verify ownership + intact Chapter 1 files!
sudo systemctl daemon-reload
sudo mount -a
sudo chown pi:pi /mnt/homelab_storage
df -h / /mnt/homelab_storage
ls -lh /mnt/homelab_storage/
```
You should see `immich_library/`, `immich_migration_dump.sql.gz`, and `immich_migration_env.bak` sitting safely on native internal PCIe NVMe inside `homelab-gpu`!

### 🖼️ Task 2.1.3 Verification: Ubuntu Server Bare-Metal Boot & Mounted Internal WD Blue SN5000

**Verification Objective:** Capture a screenshot of your terminal session on `homelab-gpu` showing `hostnamectl`, `df -h / /mnt/homelab_storage`, `ls -lh /mnt/homelab_storage/` (with `immich_migration_dump.sql.gz` visible), and `lspci | grep -E "VGA|3D"`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Unlicensed Windows 11 completely wiped from **Slot 1 (stock NVMe)** while **Slot 2 (`WD Blue SN5000`)** remained untouched.
- [ ] Ubuntu Server 24.04 LTS (`x86_64`) installed with OpenSSH enabled (and Snap Docker intentionally skipped).
- [ ] Internal **WD Blue SN5000** persistently mounted by `UUID` at `/mnt/homelab_storage` via `/etc/fstab`.
- [ ] `immich_library/` and `immich_migration_dump.sql.gz` verified intact on `homelab-gpu`.
- [ ] Discrete NVIDIA GPU verified on the PCIe bus via `lspci | grep -E "VGA|3D"`.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 2.2: Clamshell Lid-Close Mode, Power Tuning, SSH Hardening & Raspberry Pi NVMe Cleanup](./02_2_headless_lid_close_and_power_tuning.ipynb).